# QoQ-Med-VL-7B inference walkthrough

What does one inference call to QoQ-Med-VL-7B look like, and how do many of them stack into the KPSC zero-shot eval pipeline (`scripts/eval_qoq_med_kpsc800.py`)?

Everything below runs on a single synthetic image — no real MRI data needed. The pipeline pieces (`slice_to_pil_minmax`, `sample_answers`, `slice_prob_from_answers`, `aggregate`) all come from `src/qoq_eval_utils.py`, so the cells here are the exact functions the production script uses.

Run this on a GPU node (A100 / A6000). The model is ≈14 GB in bf16.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch
from PIL import Image

REPO = Path.cwd().parent          # this notebook lives in <repo>/notebooks/
sys.path.insert(0, str(REPO / "src"))
import qoq_eval_utils as qu

torch.manual_seed(0)
np.random.seed(0)

## 2. Load model + processor

Same `MODEL_ID` the eval script uses.

In [ ]:
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

MODEL_ID = "ddvd233/QoQ-Med-VL-7B"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
).eval()

print(f"params: {sum(p.numel() for p in model.parameters()) / 1e9:.1f} B")
print(f"vocab size: {model.config.vocab_size}")

## 3. Build a dummy "slice"

A 256×256 grayscale circle on a gradient — enough for the model to describe *something*. In the real pipeline this comes from `slice_to_pil_minmax(vol[s, m])` where `vol[s, m]` is one 2D modality slice of a subject's volume.

In [ ]:
H = W = 256
yy, xx = np.mgrid[:H, :W]
r = np.hypot(yy - H/2, xx - W/2)
slice2d = ((H - r) / H * 255).clip(0, 255)
slice2d += 80 * (r < 40)                          # bright blob in the middle
slice2d = slice2d.clip(0, 255).astype(np.uint8)
img = Image.fromarray(np.stack([slice2d] * 3, axis=-1), mode="RGB")
img

Equivalent to `qu.slice_to_pil_minmax(slice2d_float32)` — same per-array min/max rescale to uint8, same RGB triple.

## 4. Prompt + chat formatting

The prompt template lives in `qu.PROMPTS`. The processor's `apply_chat_template` wraps `(image, text)` in Qwen's chat format and appends an assistant-turn marker so the next token is the model's reply.

In [ ]:
prompt = qu.PROMPTS["wmd"].format(modality="T2")
print(prompt)

In [ ]:
chat = [{"role": "user", "content": [
    {"type": "image", "image": img},
    {"type": "text",  "text":  prompt},
]}]
chat_text = processor.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
print(chat_text)

In [ ]:
inputs = processor(text=[chat_text], images=[img], padding=True, return_tensors="pt").to("cuda")
{k: tuple(v.shape) for k, v in inputs.items() if hasattr(v, "shape")}

## 5. One sampled generation

`model.generate(do_sample=True, max_new_tokens=1)` returns the prompt tokens followed by one new token — we strip the prompt and decode.

In [ ]:
with torch.inference_mode():
    out = model.generate(
        **inputs,
        do_sample=True,
        temperature=0.7,
        max_new_tokens=1,
        pad_token_id=processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id,
    )
prompt_len = inputs["input_ids"].shape[1]
new_tokens = out[:, prompt_len:]
decoded = processor.tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
print("new token id:", new_tokens.tolist())
print("decoded:", decoded)

## 6. Repeat N times → slice probability

Identical to what the eval script does per slice. `qu.sample_answers` loops `model.generate` N times; `qu.slice_prob_from_answers` collapses the list of answers into a probability:

- contains `"yes"` (and no `"no"`) → 1.0
- contains `"no"`  (and no `"yes"`) → 0.0
- neither (or both) → 0.5

The slice probability is the mean over the N votes.

In [ ]:
N = 5
answers = qu.sample_answers(
    model, processor, batch=[(img, "T2")],
    n_samples=N, temperature=0.7,
    prompt_template=qu.PROMPTS["wmd"],
)[0]
print("raw answers:", answers)
print(f"slice P(Yes) = {qu.slice_prob_from_answers(answers):.3f}  (N={N})")

If you wanted higher precision, bump `N`. For the dummy image you'll see mostly 'No' — there's no anatomy that looks like white matter disease, so the model votes "No" and `P(Yes)` is near 0.

## 7. Per slice → per subject

In the real eval, a subject contributes many slices × 2 modalities (T1, T2). We collect a `P(Yes)` per slice and aggregate with `qu.aggregate`:

In [ ]:
fake_slice_probs = [0.4, 0.6, 0.8, 0.5, 0.2, 0.6, 0.7, 0.9, 0.3, 0.5]
qu.aggregate(fake_slice_probs, top_k=5)

The three numbers (`max`, `topk_mean`, `mean`) become subject-level scores. The script also computes the same three for T1 slices only and T2 slices only.

Across the full KPSC cohort, each aggregation → one column in `per_subject_scores.csv` → one (AUROC, AUPRC, balanced accuracy) entry in `metrics.json`.

## 8. Tiny end-to-end demo

Two dummy "slices" × two modalities, run the full mini-pipeline. Mirrors the inner loop of `scripts/eval_qoq_med_kpsc800.py:main` for a single subject.

In [ ]:
def make_dummy(seed):
    rng = np.random.default_rng(seed)
    s = rng.normal(0, 1, (H, W)).astype(np.float32)
    return qu.slice_to_pil_minmax(s)

dummy_slices = [(s, m, make_dummy(seed=10*s + (m == "T2")))
                for s in (0, 1) for m in ("T1", "T2")]

probs_all, mod_probs = [], {"T1": [], "T2": []}
for s, m, pil in dummy_slices:
    ans = qu.sample_answers(model, processor, [(pil, m)],
                            n_samples=3, temperature=0.7,
                            prompt_template=qu.PROMPTS["wmd"])[0]
    p = qu.slice_prob_from_answers(ans)
    print(f"slice {s} {m}: answers={ans}  P(Yes)={p:.2f}")
    probs_all.append(p)
    mod_probs[m].append(p)

print("\nsubject-level aggregations:")
print("  bag   :", qu.aggregate(probs_all, top_k=5))
print("  T1    :", qu.aggregate(mod_probs["T1"], top_k=5))
print("  T2    :", qu.aggregate(mod_probs["T2"], top_k=5))

## 9. Where this lives in the real script

Map from this notebook → `scripts/eval_qoq_med_kpsc800.py`:

| Step here | Script reference |
|---|---|
| Load volume, pick central slices | `qu.load_volume`, `qu.central_slice_indices` |
| Per-slice min/max → PIL | `qu.slice_to_pil_minmax` |
| Prompt + chat → model.generate × N | `qu.sample_answers` (in `qoq_eval_utils.py`) |
| Vote: yes/no/0.5 → mean | `qu.slice_prob_from_answers` |
| Per-subject aggregations | `qu.aggregate` |
| AUROC / AUPRC / bal_acc | `qu.metrics_for` |

To run the real eval on the cluster:

```bash
sbatch scripts/run_eval_qoq_kpsc800.slurm wmd
sbatch scripts/run_eval_qoq_kpsc800.slurm cbi
```

Artifacts land at `outputs/qoq_kpsc800_<task>_<jobid>/` and SLURM logs at `logs/qoq_kpsc800_<jobid>.{out,err}`.